In [1]:
import os
import sys

# 1. Konfigurasi lokasi cache agar tidak membebani sistem
os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"

# 2. Instalasi semua library dalam satu baris perintah
# Menggunakan {sys.executable} menjamin pip berjalan di environment yang tepat
!{sys.executable} -m pip install datasets==2.20.0 pandas transformers[torch] accelerate evaluate rouge_score bert_score -q

In [2]:
import sys
print(sys.executable)

/venv/main/bin/python


In [3]:
print(sys.path)

['/venv/main/lib/python312.zip', '/venv/main/lib/python3.12', '/venv/main/lib/python3.12/lib-dynload', '', '/venv/main/lib/python3.12/site-packages']


In [4]:
!python -m pip show datasets

/bin/bash: line 1: python: command not found


In [5]:
from datasets import load_dataset
from datasets import Dataset
import datasets
import pandas as pd
import re
import numpy as np
import os
import shutil
import torch
import re
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time
from torch.utils.data import Dataset, DataLoader
import math
import matplotlib.pyplot as plt
from tqdm.auto import tqdm 
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader

In [ ]:
import os
import sys

# 1. Konfigurasi lokasi cache
os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"

# 2. Instalasi library
!{sys.executable} -m pip install datasets==2.20.0 pandas transformers[torch] accelerate evaluate rouge_score bert_score -q

# 3. Import library
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import gc

print("--- MENGUNDUH DATASET RAW DARI HUGGING FACE ---")
# Download Meta
dataset_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Electronics", split="full", trust_remote_code=True)
df_meta = dataset_meta.to_pandas()

# Download Review
dataset_review = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Electronics", split="full", trust_remote_code=True)
df_review = dataset_review.to_pandas()

# Filter Kolom yang Diperlukan Saja (Untuk Menghemat RAM)
df_meta = df_meta[['parent_asin', 'title', 'features', 'description']]
df_review = df_review[['user_id', 'parent_asin', 'rating', 'text']]

print(f"Data Berhasil Dimuat: {len(df_meta)} Metadata, {len(df_review)} Ulasan")

--- MENGUNDUH DATASET RAW DARI HUGGING FACE ---


Generating full split:   0%|          | 0/1610012 [00:00<?, ? examples/s]

In [ ]:
print("--- MEMULAI PEMBERSIHAN DATA ---")

# 1. Membersihkan Data Ulasan (Reviews)
df_review = df_review.dropna(subset=['text', 'parent_asin', 'user_id', 'rating'])
df_review = df_review[df_review['text'].str.len() > 10]

# 2. Membersihkan Data Metadata
df_meta = df_meta.dropna(subset=['parent_asin', 'title'])

def clean_html_tags(text):
    if not isinstance(text, str):
        if isinstance(text, list): 
            text = " ".join([str(i) for i in text if i])
        else:
            return ""
    clean = re.compile('<.*?>')
    text = re.sub(clean, '', text)
    return " ".join(text.split())

df_meta['clean_description'] = df_meta['description'].apply(clean_html_tags)
df_meta['clean_features'] = df_meta['features'].apply(clean_html_tags)

print("\n--- STATUS DATA SETELAH DIBERSIHKAN ---")
print(f"Total Metadata Bersih: {len(df_meta)} baris")
print(f"Total Reviews Bersih : {len(df_review)} baris")

In [ ]:
print("--- MEMULAI DATA SPLITTING (80% WARM : 20% COLD) ---")

# 1. Ekstrak ID unik
# PERBAIKAN: Menambahkan .to_numpy() untuk MENCEGAH ERROR ArrowStringArray
unique_items = df_review['parent_asin'].unique().to_numpy()
print(f"Total Produk Unik yang memiliki ulasan: {len(unique_items)} item")

# 2. Pengacakan ID
np.random.seed(42)
np.random.shuffle(unique_items)

# 3. Hitung Split
split_idx = int(len(unique_items) * 0.8)
warm_item_ids = set(unique_items[:split_idx])
cold_item_ids = set(unique_items[split_idx:])

print(f"Jumlah Warm Items (80%): {len(warm_item_ids)} item")
print(f"Jumlah Cold Items (20%): {len(cold_item_ids)} item")

# 4. Filter Menggunakan Pandas
print("\nMemisahkan dataframe... (Proses ini aman karena RAM Anda sangat besar)")
df_review_warm = df_review[df_review['parent_asin'].isin(warm_item_ids)].copy()
df_review_cold = df_review[df_review['parent_asin'].isin(cold_item_ids)].copy()

df_meta_warm = df_meta[df_meta['parent_asin'].isin(warm_item_ids)].copy()
df_meta_cold = df_meta[df_meta['parent_asin'].isin(cold_item_ids)].copy()

print("\n--- DISTRIBUSI DATASET FINAL ---")
print(f"Data Latih Model (Warm Reviews)    : {len(df_review_warm)} baris")
print(f"Data Uji Ground Truth (Cold Reviews): {len(df_review_cold)} baris")

# 5. Membersihkan Memori Sisa (PENTING)
# Menghapus dataframe mentah agar memori kembali lega untuk proses T5 selanjutnya
del df_review
del df_meta
del dataset_meta
del dataset_review
gc.collect()

print("\n✅ Pemisahan Data Selesai & Memori telah dibersihkan!")

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
import os

print("--- 1. MENYIAPKAN DATA PELATIHAN T5 (PURE PANDAS) ---")

# 1. Gabungkan Ulasan Warm dengan Metadata Warm berdasarkan ID Produk
df_train_full = pd.merge(df_review_warm, df_meta_warm, on='parent_asin', how='inner')
print(f"Total pasangan data Warm tersedia: {len(df_train_full)}")

# 2. Ambil sampel 50.000 baris
df_t5_sample = df_train_full.sample(n=50000, random_state=42).copy()

# 3. Format kalimat Input
def format_input(row):
    title = str(row['title'])[:200]
    features = str(row['clean_features'])[:300]
    desc = str(row['clean_description'])[:300]
    return f"generate review: title: {title} features: {features} description: {desc}"

print("Memformat prompt input dan target teks...")
df_t5_sample['input_text'] = df_t5_sample.apply(format_input, axis=1)
df_t5_sample['target_text'] = df_t5_sample['text'].astype(str)

# 4. Ambil kolom penting saja & Acak Data untuk Splitting
df_t5_clean = df_t5_sample[['input_text', 'target_text']].sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Split Manual dengan Pandas (90% Train, 10% Eval) -> BYPASS ERROR DILL/PICKLER
split_idx = int(len(df_t5_clean) * 0.9)
train_df = df_t5_clean.iloc[:split_idx].reset_index(drop=True)
eval_df = df_t5_clean.iloc[split_idx:].reset_index(drop=True)

print(f"Data Siap! Train: {len(train_df)} baris | Eval: {len(eval_df)} baris")


print("\n--- 2. INISIALISASI TOKENIZER & MODEL T5 ---")
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


print("\n--- 3. MEMBUAT PYTORCH DATASET (ANTI-CRASH) ---")
# Ini adalah pengganti Hugging Face Dataset yang sering error di Python 3.14
class T5CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len_in=256, max_len_out=128):
        self.inputs = dataframe['input_text'].tolist()
        self.targets = dataframe['target_text'].tolist()
        self.tokenizer = tokenizer
        self.max_len_in = max_len_in
        self.max_len_out = max_len_out

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_text = str(self.inputs[idx])
        target_text = str(self.targets[idx])

        # Tokenisasi Input
        model_inputs = self.tokenizer(
            input_text, max_length=self.max_len_in, padding="max_length", truncation=True, return_tensors="pt"
        )
        # Tokenisasi Target
        labels = self.tokenizer(
            target_text, max_length=self.max_len_out, padding="max_length", truncation=True, return_tensors="pt"
        )

        return {
            "input_ids": model_inputs["input_ids"].squeeze(0),
            "attention_mask": model_inputs["attention_mask"].squeeze(0),
            "labels": labels["input_ids"].squeeze(0)
        }

train_dataset = T5CustomDataset(train_df, tokenizer)
eval_dataset = T5CustomDataset(eval_df, tokenizer)


print("\n--- 4. MEMULAI TRAINING T5  ---")
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-base-review-generator",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,  
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,                        
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

# 5. Simpan Model
final_model_path = "./t5-base-review-generator-final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"\n✅ Selesai! Model berhasil dilatih dan disimpan di: {final_model_path}")

In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm.auto import tqdm

print("="*60)
print("🚀 GENERASI ULASAN SINTETIS (K=5, 10, 20) - OPTIMASI EKSTREM")
print("="*60)

# 1. MEMUAT MODEL (FP16 untuk kecepatan maksimal)
model_path = "./t5-base-review-generator-final"
tokenizer_inference = T5Tokenizer.from_pretrained(model_path)
model_inference = T5ForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16 
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Mengaktifkan 2 GPU secara otomatis jika tersedia di server
if torch.cuda.device_count() > 1:
    print(f"🔥 Menggunakan {torch.cuda.device_count()} GPU untuk Inference T5!")
    model_inference = torch.nn.DataParallel(model_inference)

model_inference = model_inference.to(device)
model_inference.eval()

# 2. MENYIAPKAN PROMPT INPUT
def format_input_inference(row):
    title = str(row['title'])[:200]
    features = str(row['clean_features'])[:300]
    desc = str(row['clean_description'])[:300]
    return f"generate review: title: {title} features: {features} description: {desc}"

df_meta_cold['input_text'] = df_meta_cold.apply(format_input_inference, axis=1)
inputs = df_meta_cold['input_text'].tolist()
parent_asins = df_meta_cold['parent_asin'].tolist()

# 3. LOOPING GENERASI UNTUK SEMUA NILAI K
for K_REVIEWS in [5, 10, 20]:
    print(f"\n--- MENGHASILKAN {K_REVIEWS} ULASAN SINTETIS PER ITEM ---")
    
    # OPTIMASI 1: EKSPANSI INPUT (Anti-Bottleneck)
    expanded_inputs, expanded_asins = [], []
    for asin, inp in zip(parent_asins, inputs):
        for _ in range(K_REVIEWS):
            expanded_inputs.append(inp)
            expanded_asins.append(asin)
            
    synthetic_reviews_dict = {asin: [] for asin in parent_asins}
    results_list = []
    
    # Karena pakai FP16 & num_return_sequences=1, batch bisa disetel sangat besar
    batch_size = 256 

    for i in tqdm(range(0, len(expanded_inputs), batch_size), desc=f"T5 Menulis (K={K_REVIEWS})"):
        batch_texts = expanded_inputs[i:i+batch_size]
        
        # OPTIMASI 2: DYNAMIC PADDING ("longest")
        encoded_inputs = tokenizer_inference(
            batch_texts, padding="longest", truncation=True, max_length=256, return_tensors="pt"
        ).to(device)
        
        # OPTIMASI 3: INFERENCE MODE
        with torch.inference_mode():
            # Gunakan .module.generate jika menggunakan DataParallel (Multi-GPU)
            gen_func = model_inference.module.generate if hasattr(model_inference, 'module') else model_inference.generate
            
            generated_ids = gen_func(
                **encoded_inputs, 
                max_length=128, 
                do_sample=True, 
                temperature=0.8,
                top_p=0.9, 
                num_return_sequences=1, # <--- KUNCI KECEPATAN UTAMA
                no_repeat_ngram_size=2,
                use_cache=True
            )
            
        decoded_preds = tokenizer_inference.batch_decode(generated_ids, skip_special_tokens=True)
        results_list.extend(decoded_preds)
        
    # GABUNGKAN HASIL
    for asin, review in zip(expanded_asins, results_list):
        synthetic_reviews_dict[asin].append(review)

    # SIMPAN KE PARQUET
    df_cold_synthetic_k = pd.DataFrame({
        'parent_asin': list(synthetic_reviews_dict.keys()),
        'synthetic_reviews_list': list(synthetic_reviews_dict.values())
    })
    
    df_cold_synthetic_k['synthetic_reviews_text'] = df_cold_synthetic_k['synthetic_reviews_list'].apply(lambda x: " ".join(x))
    
    save_path = f'./cold_items_synthetic_k{K_REVIEWS}.parquet'
    df_cold_synthetic_k.to_parquet(save_path, index=False)
    print(f"✅ Selesai! File disimpan di: {save_path}")